In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import time
import scipy.optimize as sp
import skimage.io as ski 
import os
from skimage import color
from skimage import filters
from skimage import feature
from skimage.transform import hough_line, hough_line_peaks, rotate
from skimage import feature, morphology
from scipy.ndimage import binary_fill_holes
import skimage
from scipy import ndimage

import warnings
warnings.filterwarnings("ignore")
# import skimage.io as ski
# im = ski.imread("FORM1/EXAM_FORM1_62445.jpg")
# im = ski.imread("FORM1\EXAM_FORM1_61856.jpg")
# im = ski.imread("FORM1\EXAM_FORM1_62401.jpg") # image de reférence
# im = ski.imread("FORM1\EXAM_FORM1_54331.jpg") # 2 image 
# im = ski.imread("FORM1\EXAM_FORM1_54331.jpg")# illisible
im = ski.imread("FORM1\EXAM_FORM1_58220.jpg")
# im = ski.imread("FORM1\EXAM_FORM1_62380.jpeg")

if im.shape[1]>im.shape[0]:
    im = rotate(im, -90)
im_rgb = im[:, :, :3]  # on garde uniquement R, G, B
im2 = color.rgb2gray(im_rgb)

# dir(skimage.filters)
im3 = filters.sobel(im2)


size = 5
seuil = filters.threshold_otsu(im3)
im4 = im3>seuil*0.7
im4 = morphology.remove_small_objects(im4, min_size=100)
im_dilate = ndimage.binary_dilation(im4, structure=np.ones((size, size)))
im_close = morphology.binary_closing(im_dilate, morphology.square(size))
im4 = binary_fill_holes(im_close)
imlabel, nbCC   = skimage.measure.label(im4, connectivity=2, return_num= True)
regionprops= skimage.measure.regionprops(imlabel)
dimMax= 0 
l=0
L=0
i=0
for k in range(len(regionprops)):

    min_row, min_col, max_row, max_col = regionprops[k].bbox
    
    pattern = imlabel[min_row:max_row, min_col:max_col] > 0
    
    dimL, diml = pattern.shape
    if dimL*diml > dimMax and dimL<=3*diml and diml<=3*dimL and dimL*diml< 0.2*im2.shape[0]*im2.shape[1]:
        dimMax = dimL*diml
        l=diml
        L=dimL
        i=k

min_row, min_col, max_row, max_col = regionprops[i].bbox

plt.imshow(im4, cmap='gray')
plt.plot([min_col, max_col], [min_row, min_row], 'r-')  # ligne du haut
plt.plot([min_col, max_col], [max_row, max_row], 'r-')  # ligne du bas
plt.plot([min_col, min_col], [min_row, max_row], 'r-')  # ligne de gauche
plt.plot([max_col, max_col], [min_row, max_row], 'r-')  # ligne de droite
plt.title("Plus grande région")
signature = im2[min_row:max_row, min_col:max_col]


In [ ]:

h, w = signature.shape
p=25
crop = signature[p:h-p, p:w-p]

plt.imshow(crop, cmap='gray')


In [ ]:
from skimage import io, color, transform
import os
import numpy as np

def load_dataset(base_path):
    X = []
    y = []

    for person in os.listdir(base_path):
        person_path = os.path.join(base_path, person)

        if not os.path.isdir(person_path):
            continue

        for file in os.listdir(person_path):
            if file.endswith(".png"):
                path = os.path.join(person_path, file)

                img = io.imread(path)

                img = transform.resize(img, (64, 128))
                # conversion seulement si RGB
                if len(img.shape) == 3:
                    img = color.rgb2gray(img)

                img = transform.resize(img, (64, 128))
                X.append(img.flatten())
                y.append(person)

    return np.array(X), np.array(y)

In [ ]:
from sklearn.svm import SVC

print(os.listdir(r"C:\Users\frdch\Documents\Francois\Classe\ISEP\cycleIngénieur\2meAnnee\Vision par ordinateur\PROJECT 2026 - EXAM FORM RECOGNITION DATABASE-20260330\ProjectShapedetection\PROJECT 2026 -DATABASE-20260519\SIGNATURES"))
X, y = load_dataset(r"C:\Users\frdch\Documents\Francois\Classe\ISEP\cycleIngénieur\2meAnnee\Vision par ordinateur\PROJECT 2026 - EXAM FORM RECOGNITION DATABASE-20260330\ProjectShapedetection\PROJECT 2026 -DATABASE-20260519\SIGNATURES\Scan_16042026093714_SIG")
model = SVC(kernel='linear', probability=True)
model.fit(X, y)

In [ ]:
test_sig = transform.resize(signature, (64, 128)).flatten()

pred = model.predict([test_sig])[0]
proba = model.predict_proba([test_sig]).max()

print("ID prédit :", pred)
print("Confiance :", proba)

In [ ]:
if proba > 0.7:
    print("✅ Signature valide")
else:
    print("❌ Signature douteuse")

In [ ]:

def load_person_signatures(folder):
    X = []

    for file in os.listdir(folder):
        if file.endswith(".png"):
            path = os.path.join(folder, file)
            img = ski.imread(path)

            if len(img.shape) == 3:
                img = color.rgb2gray(img)

            img = transform.resize(img, (64, 128))
            X.append(img.flatten())

    return np.array(X)
person_path = r"C:\Users\frdch\Documents\Francois\Classe\ISEP\cycleIngénieur\2meAnnee\Vision par ordinateur\PROJECT 2026 - EXAM FORM RECOGNITION DATABASE-20260330\ProjectShapedetection\PROJECT 2026 -DATABASE-20260519\SIGNATURES\Scan_16042026093714_SIG\54331"

X_person = load_person_signatures(person_path)
test_sig = transform.resize(signature, (64, 128)).flatten()
from sklearn.metrics.pairwise import euclidean_distances

dists = euclidean_distances([test_sig], X_person)

min_dist = np.min(dists)

print("distance minimale :", min_dist)

if min_dist < 5:
    print("✅ Signature VALIDE (correspond à l'ID)")
else:
    print("❌ Signature NON valide")



In [ ]:
import os
import numpy as np
import skimage.io as ski
from skimage import color, filters, transform, morphology, feature
from sklearn.metrics.pairwise import cosine_similarity


# --------------------------
# ✅ 1. PREPROCESS SIGNATURE
# --------------------------
def preprocess_signature(img):

    if len(img.shape) == 3:
        img = color.rgb2gray(img)

    # normalisation lumière
    img = (img - np.mean(img)) / (np.std(img) + 1e-8)

    # binarisation
    th = filters.threshold_otsu(img)
    binary = img < th

    # enlever gros objets (cadre !)
    binary = morphology.remove_small_objects(binary, min_size=200)

    # récupérer composante principale (signature)
    labels = morphology.label(binary)
    regions = skimage.measure.regionprops(labels)

    if len(regions) == 0:
        return np.zeros((64*128,))

    largest = max(regions, key=lambda r: r.area)

    minr, minc, maxr, maxc = largest.bbox
    crop = binary[minr:maxr, minc:maxc]

    # centrage dans une image noire
    out = np.zeros((64, 128))

    h, w = crop.shape

    # resize avec conservation ratio
    scale = min(64 / h, 128 / w)
    new_h = int(h * scale)
    new_w = int(w * scale)

    resized = transform.resize(crop, (new_h, new_w))

    # centrer
    y0 = (64 - new_h)//2
    x0 = (128 - new_w)//2

    out[y0:y0+new_h, x0:x0+new_w] = resized

    return out.flatten()

# --------------------------
# ✅ 2. LOAD SIGNATURES DB
# --------------------------
def load_person_signatures(folder):
    X = []

    for file in os.listdir(folder):
        if file.endswith(".png"):
            path = os.path.join(folder, file)
            img = ski.imread(path)

            feat = preprocess_signature(img)
            X.append(feat)

    return np.array(X)


# --------------------------
# ✅ 3. COMPARISON
# --------------------------
def compare_signature(test_signature, X_person):

    test_feat = preprocess_signature(test_signature)

    sims = cosine_similarity([test_feat], X_person)

    max_sim = np.max(sims)

    print("similarité max :", max_sim)

    if max_sim > 0.22:
        print("✅ Signature VALIDE")
    else:
        print("❌ Signature NON valide")

    return max_sim


# --------------------------
# ✅ 4. EXECUTION
# --------------------------

person_path = r"C:\Users\frdch\Documents\Francois\Classe\ISEP\cycleIngénieur\2meAnnee\Vision par ordinateur\PROJECT 2026 - EXAM FORM RECOGNITION DATABASE-20260330\ProjectShapedetection\PROJECT 2026 -DATABASE-20260519\SIGNATURES\Scan_16042026093714_SIG\62276"

X_person = load_person_signatures(person_path)

# signature extraite depuis ton image
# (celle que tu avais : signature = ...)

test_signature = crop

compare_signature(test_signature, X_person)

In [ ]:
# import os

# def lister_dossier(chemin_depart):
#     for racine, dossiers, fichiers in os.walk(chemin_depart):
#         # Calcule le niveau de profondeur pour l'affichage
#         niveau = racine.replace(chemin_depart, '').count(os.sep)
#         indentation = ' ' * 4 * (niveau)
        
#         # Affiche le dossier courant
#         print(f"{indentation}[Dossier] {os.path.basename(racine)}/")
        
#         # Affiche les fichiers de ce dossier
#         indentation_fichier = ' ' * 4 * (niveau + 1)
#         for fichier in fichiers:
#             print(f"{indentation_fichier}- {fichier}")

# # Exemple d'utilisation
# lister_dossier(".")

In [ ]:
import os
import numpy as np
import skimage.io as ski
from skimage import color, filters, transform, morphology, measure
from sklearn.metrics.pairwise import cosine_similarity


# --------------------------
# ✅ PREPROCESS SIGNATURE
# --------------------------
def preprocess_signature(img):

    if len(img.shape) == 3:
        img = color.rgb2gray(img)

    # normalisation
    img = (img - np.mean(img)) / (np.std(img) + 1e-8)

    # binarisation
    th = filters.threshold_otsu(img)
    binary = img < th

    # ✅ épaissir la signature (robustesse test)
    binary = morphology.binary_dilation(binary, morphology.disk(2))

    # enlever bruit
    binary = morphology.remove_small_objects(binary, min_size=200)

    # composante principale
    labels = morphology.label(binary)
    regions = measure.regionprops(labels)

    if len(regions) == 0:
        return np.zeros((64*128,))

    largest = max(regions, key=lambda r: r.area)

    minr, minc, maxr, maxc = largest.bbox
    crop = binary[minr:maxr, minc:maxc]

    # normalisation taille
    out = np.zeros((64, 128))

    h, w = crop.shape
    scale = min(64/h, 128/w)

    new_h = int(h * scale)
    new_w = int(w * scale)

    resized = transform.resize(crop, (new_h, new_w))

    y0 = (64 - new_h)//2
    x0 = (128 - new_w)//2

    out[y0:y0+new_h, x0:x0+new_w] = resized

    feat = out.flatten()

    # ✅ normalisation vectorielle
    feat = feat / (np.linalg.norm(feat) + 1e-8)

    return feat


# --------------------------
# ✅ LOAD DATABASE
# --------------------------
def load_signature_db(base_path):

    database = {}

    for student_id in os.listdir(base_path):
        person_path = os.path.join(base_path, student_id)

        if not os.path.isdir(person_path):
            continue

        feats = []

        for file in os.listdir(person_path):
            if file.endswith(".png"):
                img = ski.imread(os.path.join(person_path, file))
                feat = preprocess_signature(img)

                # filtrage signatures vides
                if np.mean(feat) < 1e-3:
                    continue

                feats.append(feat)

        if len(feats) > 0:
            database[student_id] = np.array(feats)

    return database


# --------------------------
# ✅ MATCHING ROBUSTE
# --------------------------
def find_student_id(signature_img, database):

    test_feat = preprocess_signature(signature_img)

    best_id = None
    best_score = -1

    all_scores = []

    for student_id, X in database.items():

        sims = cosine_similarity([test_feat], X)[0]

        # ✅ top-k robuste
        top_k = min(5, len(sims))
        best_scores = np.sort(sims)[-top_k:]

        # ✅ score combiné (clé)
        score = 0.7*np.mean(best_scores) + 0.3*np.max(best_scores)

        # ✅ pénalité distance
        mean_X = np.mean(X, axis=0)
        dist = np.linalg.norm(test_feat - mean_X)
        score = score - 0.02 * dist

        all_scores.append((student_id, score))

        if score > best_score:
            best_score = score
            best_id = student_id

    # tri debug
    all_scores.sort(key=lambda x: x[1], reverse=True)

    print("\nTop 5 candidats :")
    for s in all_scores[:5]:
        print(s)

    print("\n✅ ID trouvé :", best_id)
    print("✅ Score :", best_score)

    return best_id, best_score


# --------------------------
# ✅ EXECUTION
# --------------------------
db_path = r"SIGNATURES/Scan_16042026093714_SIG"
database = load_signature_db(db_path)

img = ski.imread(r"FORM1/EXAM_FORM1_62380.jpeg")

student_id, score = find_student_id(img, database)

# ✅ décision finale
if score > 0.22:
    print("✅ Signature VALIDE")
else:
    print("❌ Signature NON valide")